In [194]:
import pandas as pd

comp = pd.read_csv("data/comp trial 1.csv", low_memory=False)
fjc = pd.read_csv("data/fjc trial 1.csv", low_memory=False)

print(comp.shape, fjc.shape)
print(comp.head())
print(fjc.head())

(332314, 35) (841, 6)
  costat curcd datafmt indfmt consol  tic    datadate  gvkey      conm  \
0      A   USD     STD   INDL      C  AIR  31/05/2000   1004  AAR CORP   
1      A   USD     STD   INDL      C  AIR  31/05/2001   1004  AAR CORP   
2      A   USD     STD   INDL      C  AIR  31/05/2002   1004  AAR CORP   
3      A   USD     STD   INDL      C  AIR  31/05/2003   1004  AAR CORP   
4      A   USD     STD   INDL      C  AIR  31/05/2004   1004  AAR CORP   

      cik  ...     rect      seq    ebit      ni      sale    xint     xsga  \
0  1750.0  ...  128.348  339.515  70.658  35.163  1024.333  23.431  102.195   
1  1750.0  ...  115.187  340.212  45.790  18.531   874.255  21.887   96.077   
2  1750.0  ...   77.528  310.235   4.711 -58.939   638.721  19.798   85.037   
3  1750.0  ...   66.322  294.988   3.573 -12.410   606.337  19.539   78.845   
4  1750.0  ...  104.661  301.684  20.811   3.504   651.958  18.819   81.165   

     capx   oancf    mkvalt  
0  22.344  10.051  372.7519 

In [195]:
comp["datadate"] = pd.to_datetime(comp["datadate"], dayfirst=True, errors="coerce")
fjc["FILEDATE"] = pd.to_datetime(fjc["FILEDATE"], dayfirst=True, errors="coerce")
comp["datadate"].head()

0   2000-05-31
1   2001-05-31
2   2002-05-31
3   2003-05-31
4   2004-05-31
Name: datadate, dtype: datetime64[us]

In [196]:
comp = comp[(comp["datadate"] >= "2000-01-01") & (comp["datadate"] <= "2018-12-31")]
fjc = fjc[(fjc["FILEDATE"] >= "2000-01-01") & (fjc["FILEDATE"] <= "2019-12-31")]

In [197]:
comp["cik"] = pd.to_numeric(comp["cik"], errors="coerce")
fjc["cik"] = pd.to_numeric(fjc["cik"], errors="coerce")

comp = comp.dropna(subset=["cik"]).copy()
fjc = fjc.dropna(subset=["cik"]).copy()

comp["cik"] = comp["cik"].astype(int)
fjc["cik"] = fjc["cik"].astype(int)

merged = comp.merge(fjc[["cik", "FILEDATE"]], on="cik", how="left")
#merge comp and add the cik and filedate columns for all shared cik values, using a left join to keep all rows from comp and add matching rows from fjc.
merged["days"] = (merged["FILEDATE"] - merged["datadate"]).dt.days
#calculate a new column "days" that shows the difference in days between the filiing date (bankruptcy) and the datadate (company's fisical year end date). 
#Positive days means the filing date is after the data date; negative means it's before.
merged["distress"] = ((merged["days"] > 0) & (merged["days"] <= 365)).astype(int)
#create a new column "distress" that is True (1) if both conditions are met: the filing date is after the data date and within 365 days, otherwise False (0).
merged.loc[merged["FILEDATE"].isna(), "distress"] = 0
#set distress to 0 for rows where FILEDATE is NaN, meaning there is no filing date available.
#loc is used here to acess the filedate column and then check (.isna()) if the value is NaN (returns True), and then explicitly sets the distress to 0. 
#Under the assumption that the data is not missing , we can assume that if there is no filing date, the company did not file for bankruptcy within the year after the datadate.
merged_clean = merged.groupby(["gvkey", "datadate"], as_index=False).agg(distress=("distress", "max"))
#this line groups the merged dataframe which has comp and fjc data merged by cik and datadate, as well as the days and distress columns( all previous commands), and groups rows into bucket (groupby) by shared gvkey and datadate values, and then aggregates the distress column by taking the maximum value for each group. This is done to ensure that if there are multiple rows for the same gvkey and datadate (redudant), we only keep one row with the highest distress value (1 if any of the rows had distress, otherwise 0). The as_index=False argument ensures that gvkey and datadate remain as columns in the resulting dataframe rather than being set as the index.
final = comp.merge(merged_clean, on=["gvkey", "datadate"], how="left")
# now we merge again to the final dataframe,  which is comp + merged+ merged_clean, to add the distress column to the og dataframe, using a left join to keep all rows from comp and add matching rows from merged_clean.
final["distress"] = final["distress"].fillna(0).astype(int)
#change all NaN values in the distress column to 0 , and convert the column to integer type, since the column after .fillna() will be float type due to the presence of NaN values.

print(len(comp), len(final))
#should be the same length , a check we didnot lose any rows in the final merge.
print(final["distress"].value_counts())
final["year"] = final["datadate"].dt.year
print(final.groupby("year")["distress"].sum())
#This is for deciding how to split the data into training and testing sets. We want to make sure that we have enough positive cases of distress in the training set to train the model effectively. If we have too few positive cases, the model may not learn to identify distress effectively, leading to poor performance on the test set.


195302 195302
distress
0    195050
1       252
Name: count, dtype: int64
year
2000     1
2001     3
2002     1
2003     0
2004     8
2005     3
2006    13
2007    26
2008    26
2009    17
2010    24
2011    23
2012    14
2013    18
2014    16
2015    25
2016    16
2017    13
2018     5
Name: distress, dtype: int64


In [198]:
print(final["datadate"].min(), final["datadate"].max())
print(final["year"].min(), final["year"].max())
print(final.groupby("year").size())  # all firm-years per year, not just distress

2000-01-31 00:00:00 2018-12-31 00:00:00
2000 2018
year
2000    12154
2001    11692
2002    11434
2003    11230
2004    10969
2005    10814
2006    10535
2007    10205
2008     9894
2009     9769
2010     9744
2011     9785
2012    10060
2013    10116
2014     9846
2015     9524
2016     9381
2017     9148
2018     9002
dtype: int64


In [199]:
#Checking categorical metadata variables in order to filter the data to only include the relevant firms.
print(final["consol"].value_counts(dropna=False)) 
print()
print(final["indfmt"].value_counts(dropna=False))
print()
print(final["datafmt"].value_counts(dropna=False))
print()
print(final["curcd"].value_counts(dropna=False))
print()
print(final["costat"].value_counts(dropna=False))

consol
C    195302
Name: count, dtype: int64

indfmt
INDL    172673
FS       22629
Name: count, dtype: int64

datafmt
STD    195302
Name: count, dtype: int64

curcd
USD    179833
CAD     15469
Name: count, dtype: int64

costat
I    102188
A     93114
Name: count, dtype: int64


In [200]:
#For the metadata variables, we want to filter the data to only include firms that are eligible for analysis
final = final[final["indfmt"] == "INDL"] #limiting to industrial format firms, which are the most common type of firms in the dataset and are more likely to have complete financial data.
final = final[final["curcd"] == "USD"] #limiting to firms that report in USD, so that we can compare financial data across firms without having to worry about currency conversion.
# we keep both I and A values for the costat column, since we want to include both active and inactive firms in the analysis.

print(final.columns.tolist())

['costat', 'curcd', 'datafmt', 'indfmt', 'consol', 'tic', 'datadate', 'gvkey', 'conm', 'cik', 'fyr', 'fic', 'loc', 'sic', 'act', 'at', 'ceq', 'che', 'dlc', 'dltt', 'invt', 'lct', 'lt', 'ppent', 're', 'rect', 'seq', 'ebit', 'ni', 'sale', 'xint', 'xsga', 'capx', 'oancf', 'mkvalt', 'distress', 'year']


In [201]:
#Looking at the amount of missing values in the features columns
print(final["act"].isna().sum()/len(final))
print(final["at"].isna().sum()/len(final))
print(final["ceq"].isna().sum()/len(final))
print(final["che"].isna().sum()/len(final))
print(final["dlc"].isna().sum()/len(final))
print(final["dltt"].isna().sum()/len(final))
print(final["invt"].isna().sum()/len(final))
print(final["lct"].isna().sum()/len(final))
print(final["lt"].isna().sum()/len(final))
print(final["ppent"].isna().sum()/len(final))
print(final["rect"].isna().sum()/len(final))
print(final["re"].isna().sum()/len(final))
print(final["ebit"].isna().sum()/len(final))
print(final["seq"].isna().sum()/len(final))
print(final["ni"].isna().sum()/len(final))
print(final["sale"].isna().sum()/len(final))
print(final["xint"].isna().sum()/len(final))
print(final["xsga"].isna().sum()/len(final))
print(final["capx"].isna().sum()/len(final))
print(final["oancf"].isna().sum()/len(final))
print(final["mkvalt"].isna().sum()/len(final))


0.2677802542930956
0.09517628286307685
0.09746213464363507
0.0953029228509194
0.09606276277797478
0.09766475862418317
0.10554176586799048
0.264981510561775
0.09675928271110885
0.11938351653918242
0.1013499822704017
0.12522795197811662
0.102996302112355
0.09521427485942961
0.09893115850260878
0.0989564865001773
0.2360759333367104
0.2542234435945494
0.12920444759637303
0.12493034800668659
0.28587077655640547


In [202]:
features = ["act", "at", "ceq", "che", "dlc", "dltt", "invt", "lct", "lt", 
            "ppent", "rect", "re", "ebit", "seq", "ni", "sale", "xint", 
            "xsga", "capx", "oancf", "mkvalt"]
#Main accounting variables.

final.groupby("distress")[features].apply(lambda x: x.isna().mean())
#Some checks to see how many missing values there are in the features columns, grouped by distress.
#final.groupby("distress")[features].apply(lambda x: x.isna().mean().max())
#print(final["distress"].value_counts())



,act,at,ceq,che,dlc,dltt,invt,lct,lt,ppent,...,re,ebit,seq,ni,sale,xint,xsga,capx,oancf,mkvalt
distress,,,,,,,,,,,,,,,,,,,,,
0,0.268031,0.095302,0.097592,0.095429,0.096190,0.097795,0.105658,0.265228,0.096888,0.119532,...,0.125322,0.103127,0.095340,0.099063,0.099088,0.236344,0.254379,0.129342,0.125075,0.286053
1,0.095652,0.008696,0.008696,0.008696,0.008696,0.008696,0.026087,0.095652,0.008696,0.017391,...,0.060870,0.013043,0.008696,0.008696,0.008696,0.052174,0.147826,0.034783,0.026087,0.160870


In [ ]:
#For missing values, we make a missing value indicator column for each feature, which is 1 if the value is missing and 0 if it is not missing. This way, we can use the missing value information as a feature in the model, which may be informative for predicting distress.

for feature in features:
    final[feature + "_missing"] = final[feature].isna().astype(int)

final.columns.tolist() #check
final[["at_missing", "act_missing","capx_missing"]].tail() #check rows

 #Now, to get the median to be a little more accurate compared to the overall median for firms.
 #I group firms by shared sic and fiscal year and then based each unique combination a median is calculated and its used to fill in the missing cells.
industry_categories = ["sic","year"]
features_medians = final.groupby(industry_categories, as_index=False)[features].transform('median')
final[features] = final[features].fillna(features_medians)

final[features].isna().sum()

 

act       13431
at           50
ceq          50
che          50
dlc          50
dltt         50
invt         50
lct       13201
lt           50
ppent        50
rect         50
re           78
ebit         50
seq          50
ni           50
sale         50
xint       3173
xsga       1706
capx        790
oancf       789
mkvalt      170
dtype: int64